# Comparing Model Behavior: Base vs Fine-Tuned vs RL

This lab compares how three versions of the same model respond to the same math prompt:
- **Base model** – raw pretrained, no instruction tuning
- **Fine-tuned (SFT)** – instruction-tuned via supervised fine-tuning
- **RL model** – further trained with reinforcement learning

Works on **Mac (Apple Silicon / MPS)** and **Google Colab (CUDA)**.

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("transformers", "accelerate", "sentencepiece", "protobuf")

# bitsandbytes only works on CUDA — skip on Mac
import torch
if torch.cuda.is_available():
    pip_install("bitsandbytes")
    print("Installed bitsandbytes for CUDA quantization")
else:
    print("Skipping bitsandbytes (not on CUDA)")

## Cell 2 — Detect environment

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    import psutil
    print(f"Apple Silicon MPS")
    print(f"System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
else:
    DEVICE = "cpu"
    print("Running on CPU (will be slow)")

print(f"\nUsing device: {DEVICE}")

## Cell 3 — Config

In [ ]:
BASE_MODEL      = "deepseek-ai/deepseek-math-7b-base"
FINETUNED_MODEL = "deepseek-ai/deepseek-math-7b-instruct"
RL_MODEL        = "deepseek-ai/deepseek-math-7b-rl"

EXAMPLE_PROMPT = (
    "Find the smallest positive integer N such that: "
    "N leaves remainder 1 when divided by 2, remainder 2 when divided by 3, "
    "remainder 3 when divided by 4, and so on, up to remainder 9 when divided by 10. "
    "N is divisible by 11"
)

MAX_NEW_TOKENS = 512

print("Prompt:")
print(EXAMPLE_PROMPT)

## Cell 4 — Helper functions

- On **CUDA**: loads in 4-bit (saves ~10 GB VRAM vs float16)
- On **MPS / CPU**: loads in float16 (bitsandbytes not supported)

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


def load_model(model_id: str):
    """Load model + tokenizer. Uses 4-bit on CUDA, float16 on MPS."""
    print(f"Loading {model_id} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if DEVICE == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto",   # accelerate handles MPS/CPU mapping
        )

    model.eval()
    print(f"  Loaded. Device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else DEVICE}")
    return model, tokenizer


def unload_model(model):
    """Free model memory before loading the next one."""
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()
    print("  Model unloaded.\n")


def make_base_prompt(question: str) -> str:
    """Base model: raw completion — no chat template."""
    return question


def make_instruct_prompt(question: str) -> str:
    """Instruct / RL model: DeepSeek-Math chat format."""
    return (
        f"User: {question}\n\n"
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        "Assistant:"
    )


@torch.inference_mode()
def generate(model, tokenizer, prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # greedy — deterministic
        temperature=1.0,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Return only the newly generated tokens (strip the prompt)
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("Helper functions ready.")

## Cell 5 — Run BASE model

Expect raw text continuation — no structured reasoning.

In [ ]:
base_model, base_tok = load_model(BASE_MODEL)

base_prompt = make_base_prompt(EXAMPLE_PROMPT)
print(f"Prompt sent:\n{base_prompt}\n")
print("-" * 60)

base_output = generate(base_model, base_tok, base_prompt)
print("BASE MODEL OUTPUT:")
print(base_output)

unload_model(base_model)

## Cell 6 — Run FINE-TUNED (SFT / Instruct) model

Expect structured step-by-step reasoning and a boxed answer.

In [ ]:
sft_model, sft_tok = load_model(FINETUNED_MODEL)

sft_prompt = make_instruct_prompt(EXAMPLE_PROMPT)
print(f"Prompt sent:\n{sft_prompt}\n")
print("-" * 60)

sft_output = generate(sft_model, sft_tok, sft_prompt)
print("FINE-TUNED (SFT) MODEL OUTPUT:")
print(sft_output)

unload_model(sft_model)

## Cell 7 — Run RL model

Expect more rigorous step-by-step reasoning — RL reward signal pushed it toward correctness.

In [ ]:
rl_model, rl_tok = load_model(RL_MODEL)

rl_prompt = make_instruct_prompt(EXAMPLE_PROMPT)
print(f"Prompt sent:\n{rl_prompt}\n")
print("-" * 60)

rl_output = generate(rl_model, rl_tok, rl_prompt)
print("RL MODEL OUTPUT:")
print(rl_output)

unload_model(rl_model)

## Cell 8 — Side-by-side comparison

In [ ]:
SEPARATOR = "=" * 70

print(SEPARATOR)
print("PROMPT")
print(SEPARATOR)
print(EXAMPLE_PROMPT)

print(f"\n{SEPARATOR}")
print("1. BASE MODEL  (deepseek-math-7b-base)")
print(SEPARATOR)
print(base_output)

print(f"\n{SEPARATOR}")
print("2. FINE-TUNED  (deepseek-math-7b-instruct — SFT)")
print(SEPARATOR)
print(sft_output)

print(f"\n{SEPARATOR}")
print("3. RL MODEL    (deepseek-math-7b-rl)")
print(SEPARATOR)
print(rl_output)

print(f"\n{SEPARATOR}")
print("KEY OBSERVATIONS TO LOOK FOR:")
print(SEPARATOR)
print("""
Base model  → likely raw continuation, no structured reasoning
SFT model   → follows instruction format, attempts step-by-step
RL model    → more rigorous reasoning, more likely to reach correct \\boxed{} answer

Correct answer: N = 2519  (LCM(2..10) - 1 = 2520 - 1 = 2519, and 2519 / 11 = 229)
""")